# Pillar 5 - Phase 1 baseline: retriever near-duplicate discrimination (5.1)

See `plans/PLAN.md` (Tru cot 5, question 5.1) and `phases/phase-1-no-train.md`.

Tests whether a Vietnamese dense retriever can tell apart two near-identical passages
that differ in one key fact (a number, year, or category) - not just topical retrieval.
Runs on Kaggle CPU (inference only, no training).


In [ ]:
import torch, json, statistics
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


In [ ]:
# (query, correct_passage, distractor_passage) - correct/distractor differ in one key fact
triples = [
    ("Lãi suất tiết kiệm kỳ hạn 12 tháng năm 2025 là bao nhiêu?",
     "Theo biểu lãi suất niêm yết, tiết kiệm kỳ hạn 12 tháng năm 2025 là 5.2%/năm.",
     "Theo biểu lãi suất niêm yết, tiết kiệm kỳ hạn 12 tháng năm 2024 là 5.2%/năm."),
    ("Tỷ giá USD/VNĐ ngày 1/1/2026 là bao nhiêu?",
     "Ngân hàng nhà nước công bố tỷ giá trung tâm ngày 1/1/2026 là 24.500 đồng/USD.",
     "Ngân hàng nhà nước công bố tỷ giá trung tâm ngày 1/1/2025 là 24.500 đồng/USD."),
    ("Dân số tỉnh Nghệ An năm 2024 là bao nhiêu?",
     "Theo thống kê năm 2024, dân số tỉnh Nghệ An khoảng 3,4 triệu người.",
     "Theo thống kê năm 2024, dân số tỉnh Hà Tĩnh khoảng 3,4 triệu người."),
    ("Điểm chuẩn ngành Công nghệ thông tin năm 2025 là bao nhiêu?",
     "Năm 2025, điểm chuẩn ngành Công nghệ thông tin của trường là 26.5 điểm.",
     "Năm 2025, điểm chuẩn ngành Công nghệ thông tin của trường là 24.5 điểm."),
    ("Đội tuyển nào vô địch AFF Cup 2024?",
     "Đội tuyển Việt Nam giành chức vô địch AFF Cup 2024 sau khi thắng ở trận chung kết.",
     "Đội tuyển Thái Lan giành chức vô địch AFF Cup 2024 sau khi thắng ở trận chung kết."),
    ("Nhiệt độ trung bình ở Hà Nội vào tháng 1 là bao nhiêu?",
     "Vào tháng 1, nhiệt độ trung bình ở Hà Nội thường dao động quanh mức 17 độ C.",
     "Vào tháng 7, nhiệt độ trung bình ở Hà Nội thường dao động quanh mức 17 độ C."),
    ("Nghị định nào quy định xử phạt vi phạm giao thông đường bộ năm 2024?",
     "Nghị định 100/2019/NĐ-CP sửa đổi năm 2024 quy định xử phạt vi phạm giao thông đường bộ.",
     "Nghị định 100/2019/NĐ-CP sửa đổi năm 2024 quy định xử phạt vi phạm giao thông đường thủy."),
    ("Giá vàng SJC ngày 1/9/2026 là bao nhiêu?",
     "Giá vàng SJC niêm yết ngày 1/9/2026 ở mức 82 triệu đồng/lượng.",
     "Giá vàng SJC niêm yết ngày 1/8/2026 ở mức 82 triệu đồng/lượng."),
    ("Sự kiện Việt Nam thống nhất đất nước diễn ra năm nào?",
     "Việt Nam thống nhất đất nước vào ngày 30 tháng 4 năm 1975.",
     "Việt Nam thống nhất đất nước vào ngày 30 tháng 4 năm 1954."),
    ("Khoảng cách từ Hà Nội đến Đà Nẵng là bao nhiêu km?",
     "Khoảng cách đường bộ từ Hà Nội đến Đà Nẵng khoảng 760 km.",
     "Khoảng cách đường bộ từ Hà Nội đến Thành phố Hồ Chí Minh khoảng 760 km."),
    ("Giá xăng RON95 ngày 15/8/2026 là bao nhiêu?",
     "Giá bán lẻ xăng RON95 từ ngày 15/8/2026 là 22.000 đồng/lít.",
     "Giá bán lẻ xăng RON95 từ ngày 15/7/2026 là 22.000 đồng/lít."),
    ("Tuổi nghỉ hưu của lao động nam theo lộ trình năm 2028 là bao nhiêu?",
     "Theo lộ trình, tuổi nghỉ hưu của lao động nam vào năm 2028 là 62 tuổi.",
     "Theo lộ trình, tuổi nghỉ hưu của lao động nữ vào năm 2028 là 62 tuổi."),
    ("Thuế suất VAT phổ thông hiện nay là bao nhiêu phần trăm?",
     "Thuế suất giá trị gia tăng (VAT) phổ thông hiện nay là 10%.",
     "Thuế suất giá trị gia tăng (VAT) phổ thông hiện nay là 8%."),
    ("Dân số Việt Nam năm 2024 là bao nhiêu?",
     "Theo tổng cục thống kê, dân số Việt Nam năm 2024 khoảng 100,3 triệu người.",
     "Theo tổng cục thống kê, dân số Việt Nam năm 2014 khoảng 100,3 triệu người."),
    ("Mức lương tối thiểu vùng 1 hiện nay là bao nhiêu?",
     "Mức lương tối thiểu vùng 1 áp dụng hiện nay là 4.960.000 đồng/tháng.",
     "Mức lương tối thiểu vùng 4 áp dụng hiện nay là 4.960.000 đồng/tháng."),
    ("Kỳ thi tốt nghiệp THPT năm 2026 diễn ra vào tháng nào?",
     "Kỳ thi tốt nghiệp THPT năm 2026 dự kiến diễn ra vào cuối tháng 6.",
     "Kỳ thi tốt nghiệp THPT năm 2025 dự kiến diễn ra vào cuối tháng 6."),
    ("Sân bay quốc tế lớn nhất miền Nam Việt Nam là sân bay nào?",
     "Sân bay quốc tế Tân Sơn Nhất là sân bay lớn nhất miền Nam Việt Nam.",
     "Sân bay quốc tế Nội Bài là sân bay lớn nhất miền Nam Việt Nam."),
    ("GDP Việt Nam năm 2025 tăng trưởng bao nhiêu phần trăm?",
     "Theo báo cáo, GDP Việt Nam năm 2025 tăng trưởng khoảng 6.5%.",
     "Theo báo cáo, GDP Việt Nam năm 2023 tăng trưởng khoảng 6.5%."),
    ("Cầu Nhật Tân bắc qua sông nào?",
     "Cầu Nhật Tân là cây cầu bắc qua sông Hồng, nối trung tâm Hà Nội với sân bay Nội Bài.",
     "Cầu Nhật Tân là cây cầu bắc qua sông Đuống, nối trung tâm Hà Nội với sân bay Nội Bài."),
    ("Bảo hiểm y tế học sinh sinh viên năm học 2026-2027 có mức đóng bao nhiêu?",
     "Mức đóng bảo hiểm y tế học sinh sinh viên năm học 2026-2027 là khoảng 900.000 đồng/năm.",
     "Mức đóng bảo hiểm y tế học sinh sinh viên năm học 2025-2026 là khoảng 900.000 đồng/năm."),
]
print("n triples:", len(triples))


In [ ]:
@torch.no_grad()
def embed(texts, tok, model):
    enc = tok(texts, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
    out = model(**enc).last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1).float()
    pooled = (out * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
    return F.normalize(pooled, dim=-1)

def eval_retriever(model_name, triples):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(DEVICE).eval()

    correct, margins = 0, []
    for query, right, wrong in triples:
        emb = embed([query, right, wrong], tok, model)
        sim_right = (emb[0] @ emb[1]).item()
        sim_wrong = (emb[0] @ emb[2]).item()
        margins.append(sim_right - sim_wrong)
        if sim_right > sim_wrong:
            correct += 1

    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return {
        "accuracy": correct / len(triples),
        "mean_margin": statistics.mean(margins),
        "stdev_margin": statistics.stdev(margins),
        "n": len(triples),
    }

results_5_1 = {}
for name in ["bkai-foundation-models/vietnamese-bi-encoder", "dangvantuan/vietnamese-embedding"]:
    results_5_1[name] = eval_retriever(name, triples)

for k, v in results_5_1.items():
    print(k, "->", v)


In [ ]:
import os
out_dir = "/kaggle/working"
os.makedirs(out_dir, exist_ok=True)
with open(os.path.join(out_dir, "metrics_pillar5_5_1.jsonl"), "w") as f:
    f.write(json.dumps({"section": "5.1", "results": results_5_1}) + "\n")
print("wrote", os.path.join(out_dir, "metrics_pillar5_5_1.jsonl"))
